In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

class SparkConfig :
    def creat_sparksession() :
        spark = SparkSession.builder.config("spark.driver.memory" , "8g") \
                            .appName("clean data") \
                            .master("local[*]") \
                            .getOrCreate()
        spark.sparkContext._jsc.hadoopConfiguration().set("fs.defaultFS", "file:///")
        return spark

In [2]:
spark = SparkConfig.creat_sparksession()

In [3]:
class DataLoader :
    def __init__(selt) :
        selt.spark = SparkConfig.create_sparksession() 
    def clean_works() :    
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/works/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        list_file = []
        list_col = []
        for file in files :
            works = spark.read.format("parquet").load(file) 
            
            works = works.withColumn("cover_id" , col("cover_id").cast("long"))
            for c in works.columns :
                if c not in list_col :
                    list_col.append(c)
            for c in list_col :
                if c not in works.columns :
                    works = works.withColumn(c , lit(None).cast("string"))
            list_file.append(works)
        works = reduce(lambda a,b : a.union(b) , list_file)
        return works

    def clean_editions() :    
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/editions/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        list_file = []
        list_col = []
        for file in files :
            editions = spark.read.format("parquet").load(file) 
            if "number_of_pages" in editions.columns :
                editions = editions.withColumn("number_of_pages" , col("number_of_pages").cast("long"))
            
            for c in editions.columns :
                if c not in list_col :
                    list_col.append(c)
            for c in list_col :
                if c not in editions.columns :
                    editions = editions.withColumn(c , lit(None).cast("string"))
            list_file.append(editions)
        editions = reduce(lambda a,b : a.union(b) , list_file)
        editions.limit(5).coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/editions/")
        return editions 
    
    def clean_works() :        
        s3 = boto3.client("s3")
        response = s3.list_objects_v2(Bucket = "mhai-bk" , Prefix = "raw_data/authors/")
        files = [f"s3://mhai-bk/{item['Key']}" for item in response['Contents']]
        for file in files :
            authors = spark.read.format("parquet").load(file) 
            if "number_of_pages" in authors.columns :
                authors = authors.withColumn("number_of_pages" , col("number_of_pages").cast("long"))
            
            list_file.append(authors)
        authors = reduce(lambda a,b : a.unionByName(b , allowMissingColumns = True) , list_file)
        authors.limit(5).coalesce(1).write.mode("overwrite").parquet("s3://mhai-bk/data/authors/")
        return authors

In [4]:
spark

In [5]:
works = spark.read.parquet("data/works.parquet")

In [6]:
editions = spark.read.parquet("data/editions.parquet")

In [7]:
authors = spark.read.parquet("data/authors.parquet")

In [8]:
# Chọn các cột cần để phân tích
works = works.select("key", "title", "first_publish_year", "edition_count", "subject", "authors" )

In [9]:
works.printSchema()

root
 |-- key: string (nullable = true)
 |-- title: string (nullable = true)
 |-- first_publish_year: double (nullable = true)
 |-- edition_count: long (nullable = true)
 |-- subject: string (nullable = true)
 |-- authors: string (nullable = true)



In [10]:
works.select([count(when (col(c).isNull() , c)).alias(c) for c in works.columns ]).show()

+---+-----+------------------+-------------+-------+-------+
|key|title|first_publish_year|edition_count|subject|authors|
+---+-----+------------------+-------------+-------+-------+
|  0|    0|               145|            0|      0|      0|
+---+-----+------------------+-------------+-------+-------+



In [11]:
works.show()

+-----------------+--------------------+------------------+-------------+--------------------+--------------------+
|              key|               title|first_publish_year|edition_count|             subject|             authors|
+-----------------+--------------------+------------------+-------------+--------------------+--------------------+
|  /works/OL21177W|   Wuthering Heights|            1846.0|         2886|["form:novel", "g...|[{"key": "/author...|
|  /works/OL66513W|                Emma|            1815.0|         2263|["Social life and...|[{"key": "/author...|
|  /works/OL66562W|Sense and Sensibi...|            1811.0|         2090|["Fiction, Romanc...|[{"key": "/author...|
|  /works/OL29983W|        Little Women|            1848.0|         1888|["Romans", "Jeune...|[{"key": "/author...|
| /works/OL267096W|       Анна Каренина|            1876.0|         1314|["Fiction", "Adul...|[{"key": "/author...|
|/works/OL1095427W|           Jane Eyre|            1847.0|         1170

In [12]:
# Làm sạch cột key và first_pubnlish_year 
work_key = works.withColumn("key" , regexp_extract(col("key") , r"[A-Z]+\d+[A-Z]", 0)) \
                .withColumnRenamed("key", "work_id") \
                .withColumn("first_publish_year" , col("first_publish_year").cast("int")) # chuyển đổi kiểu dữ liệu cột first_publish_year

In [13]:
# Xử lý giá trị missing value bằng cách lấy trung vị
median = work_key.approxQuantile("first_publish_year", [0.5], 0.01)[0]
work_key = work_key.fillna({"first_publish_year" : int(median)})

In [14]:
# Tạo ra bảng dim_work 
dim_work = work_key.select("work_id", "title", "first_publish_year", "edition_count").dropDuplicates()

In [ ]:
# Explode cột subject 
work_explode_sub = work_key.withColumn("subject" , regexp_replace("subject" , r"[\]\[]", "")) \
                           .withColumn("subject", explode(split("subject" , ",")))

In [16]:
# Làm sạch cột subject
work_explode_sub = work_explode_sub.withColumn("subject", regexp_replace("subject", r'form:|genre:|"', ""))

In [50]:
# Làm sạch bảng work_explode_sub
work_explode_sub = work_explode_sub.filter(col("subject").isNotNull() & ~col("subject").rlike("\\\\|[0-9]"))
# Xóa bỏ khoảng trắng, lọc các dữ liệu sạch
work_explode_sub = work_explode_sub.orderBy("subject") \
           .withColumn("subject", trim(col("subject"))) \
           .filter(~col("subject").rlike("&|.com$|@|^:|^\\.") & trim(col("subject") != "") ) \
           .withColumn("subject", regexp_replace("subject" ,"'|\\*|-" , "")) 

In [51]:
# Tạo bảng dim_subject
dim_subject = work_explode_sub.select("subject").dropDuplicates()

In [53]:
dim_subject = dim_subject.distinct()

In [54]:
dim_subject.count()

43972

In [55]:
# Gen Id cho bảng dim_subject sử dụng window function
dim_subject = dim_subject.withColumn("subject_id", row_number().over(Window.orderBy("subject")))

In [56]:
dim_subject.show(5)

+---------------+----------+
|        subject|subject_id|
+---------------+----------+
|(Computer file)|         1|
| (Edwin Albert)|         2|
|   (James Yimm)|         3|
|        (Musik)|         4|
|(Robert Edward)|         5|
+---------------+----------+
only showing top 5 rows


In [63]:
# Tạo bảng bridge work_subject
work_subject = work_explode_sub.join(dim_subject , on = "subject", how = "inner") \
                .select("work_id", "subject_id").orderBy("subject")

In [66]:
work_subject.show(5)

+----------+----------+
|   work_id|subject_id|
+----------+----------+
|OL2582567W|         1|
|OL8887370W|         1|
|OL4327793W|         2|
|OL3018177W|         3|
|  OL28809W|         4|
+----------+----------+
only showing top 5 rows


In [69]:
work_explode_sub.select("authors").show(5, truncate = False)

+----------------------------------------------------------------+
|authors                                                         |
+----------------------------------------------------------------+
|[{"key": "/authors/OL184781A", "name": "Giles Lytton Strachey"}]|
|[{"key": "/authors/OL24461A", "name": "Rudyard Kipling"}]       |
|[{"key": "/authors/OL1234665A", "name": "J. J. Scarisbrick"}]   |
|[{"key": "/authors/OL19597A", "name": "Nicholas Sparks"}]       |
|[{"key": "/authors/OL184781A", "name": "Giles Lytton Strachey"}]|
+----------------------------------------------------------------+
only showing top 5 rows


In [ ]:
# Tạo author schema
author_schema = ArrayType(
    StructType([
        StructField("key", StringType(), True),
        StructField("name", StringType(), True)
    ])
)

In [91]:
# Trích ra cột id và name của author
work_author = work_explode_sub.withColumn("authors" , from_json(col("authors"), author_schema)) \
                .withColumn("authors", explode(col("authors"))) \
                .select("work_id","authors.key")

In [92]:
work_author.show(5)

+----------+-------------------+
|   work_id|                key|
+----------+-------------------+
|OL1652426W| /authors/OL184781A|
|  OL19870W|  /authors/OL24461A|
|OL5361840W|/authors/OL1234665A|
|  OL54797W|  /authors/OL19597A|
|OL1652426W| /authors/OL184781A|
+----------+-------------------+
only showing top 5 rows


In [94]:
# Làm sạch cột author_id
work_author = work_author.withColumn("key", regexp_extract(col("key"), r"[A-Z]+\d+[A-Z]", 0))

## Edition

In [58]:
editions = editions.select("key", "works", "title", "publish_date", "publishers", "languages", "number_of_pages", "isbn_13")

In [62]:
works.withColumn("subject" , col("subject").cast("array<string>")) \
     .withColumn("subject_exp" , explode(col("subject"))).select("subject_exp").show(5)

{"ts": "2026-05-05 13:12:47.449", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve \"subject\" due to data type mismatch: cannot cast \"STRING\" to \"ARRAY<STRING>\". SQLSTATE: 42K09", "context": {"file": "line 1 in cell [62]", "line": "", "fragment": "cast", "errorClass": "DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o55.withColumn.\n: org.apache.spark.sql.AnalysisException: [DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve \"subject\" due to data type mismatch: cannot cast \"STRING\" to \"ARRAY<STRING>\". SQLSTATE: 42K09;\n'Project [key#0, title#1, first_publish_year#11, edition_count#2L, cast(subject#5 as array<string>) AS subject#829, authors#10]\n+- Project [key#0, title#1, first_publish_year#11, edition_count#2L, subject#5, authors#10]\n   +- Relation [key#0,title#1,edition_count#2L,cover_id#3L,cover_edition_key#4,su

AnalysisException: [DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve "subject" due to data type mismatch: cannot cast "STRING" to "ARRAY<STRING>". SQLSTATE: 42K09;
'Project [key#0, title#1, first_publish_year#11, edition_count#2L, cast(subject#5 as array<string>) AS subject#829, authors#10]
+- Project [key#0, title#1, first_publish_year#11, edition_count#2L, subject#5, authors#10]
   +- Relation [key#0,title#1,edition_count#2L,cover_id#3L,cover_edition_key#4,subject#5,ia_collection#6,printdisabled#7,lending_edition#8,lending_identifier#9,authors#10,first_publish_year#11,ia#12,public_scan#13,has_fulltext#14,availability#15] parquet


In [ ]:
works.count()

31901

In [ ]:
editions.count()

224540

In [ ]:
editions.select([count(when (col(c).isNull() , c)).alias(c) for c in editions.columns ]).show()

+---+-----+-----+------------+----------+---------+---------------+-------+
|key|works|title|publish_date|publishers|languages|number_of_pages|isbn_13|
+---+-----+-----+------------+----------+---------+---------------+-------+
|  0|    0|    9|        2957|      1978|    27963|         116486|  59642|
+---+-----+-----+------------+----------+---------+---------------+-------+



In [ ]:
authors.select([count(when (col(c).isNull() , c)).alias(c) for c in authors.columns ]).show()

+---+----+----------+
|key|name|birth_date|
+---+----+----------+
|  0|1351|    114660|
+---+----+----------+



In [ ]:
authors.count()

125500